# 6 · The solver toolbox 🛠

Every `Inverse(...)` so far hid a whole field. Here are the standard building blocks — a
**direct** solver, the **iterative** Krylov methods (**CG**, **GMRes**), and the
**preconditioners** that make them converge — plus **threads**, a **profile** and **timers**.
We only show the interface; the i-tutorials carry the analysis.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from ngsolve import solvers
import sys

if sys.platform != "emscripten":                     # JupyterLite (WebAssembly) is single-threaded
    SetNumThreads(4)                                  # use up to 4 cores (see §3)
mesh = Mesh(unit_square.GenerateMesh(maxh=0.08))
fes = H1(mesh, order=3, dirichlet=".*")
u, v = fes.TnT()
f = LinearForm(v * dx).Assemble()
print(f"test problem: Poisson, {fes.ndof} dofs")

## 1. The direct solver — and a reminder about free dofs

A **direct** solver factorises the matrix once and back-substitutes — robust and exact up to
round-off (`sparsecholesky` is the SPD one that also runs in WebAssembly). As in unit 5 we
invert only on the **free dofs**. It is unbeatable until the **factorisation** no longer fits
in memory; then we go iterative.

In [ ]:
a = BilinearForm(grad(u) * grad(v) * dx).Assemble()
gfu = GridFunction(fes)
gfu.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f.vec
print(f"direct (sparsecholesky): solution max = {max(gfu.vec):.4f}")

## 2. Iterative solvers & preconditioners

A Krylov solver (**`CG`** for symmetric positive-definite systems, **`GMRes`** otherwise) never
forms an inverse — it only multiplies by $A$ and needs a cheap **preconditioner**
$C\approx A^{-1}$. In NGSolve you **register** a `Preconditioner(a, name)` on the form *before*
assembly and hand it to the solver. One helper does exactly that — and, to see how a
preconditioner behaves under **mesh refinement**, it refines a few times and reports the CG
step count `inv.GetSteps()`:

In [ ]:
def SolveProblem(precond="local", levels=4, p=3, h=0.3):
    mesh = Mesh(unit_square.GenerateMesh(maxh=h))
    fes = H1(mesh, order=p, dirichlet=".*", autoupdate=True)
    u, v = fes.TnT()
    a = BilinearForm(grad(u) * grad(v) * dx)
    pre = Preconditioner(a, precond)                 # register BEFORE assembly
    f = LinearForm(v * dx)
    gfu = GridFunction(fes, autoupdate=True)
    steps = []
    for l in range(levels):
        if l: mesh.Refine()
        a.Assemble(); f.Assemble()
        inv = CGSolver(a.mat, pre.mat, printrates=False, precision=1e-8, maxsteps=1000)
        gfu.vec.data = inv * f.vec
        steps.append((fes.ndof, inv.GetSteps()))
    return steps, gfu

In [ ]:
for precond in ["local", "multigrid", "bddc"]:       # "local" = point Jacobi
    steps, gfu = SolveProblem(precond)
    print(f"  {precond:10s}: " + "  ".join(f"{nd:>6d} dofs ->{it:3d} it" for nd, it in steps))
Draw(gfu, gfu.space.mesh, "u (finest level)")

Same `CG`, same problem — only the `precond` string changed, yet the step counts behave very
differently as the mesh is refined. *Which* preconditioner scales how — and others like
`"bddc"`, block-Jacobi or AMG — is the subject of **i-tutorial 2.1.1**; here we only show the
interface. For a **non-symmetric** system `CG` is invalid and **`GMRes`** takes over with the
very same preconditioner idea:

In [ ]:
anonsym = BilinearForm(grad(u) * grad(v) * dx + (CF((30, 0)) * grad(u)) * v * dx)   # + convection
pre_ns = Preconditioner(anonsym, "local"); anonsym.Assemble()
gns = GridFunction(fes)
solvers.GMRes(A=anonsym.mat, b=f.vec, x=gns.vec, pre=pre_ns.mat, tol=1e-8, maxsteps=1000, printrates=False)
print(f"GMRes (convection-diffusion, non-symmetric): max = {max(gns.vec):.4f}")

## 3. Threads, a profile, and timers

**`SetNumThreads(n)`** sets the task parallelism; a **`TaskManager`** block farms the work
across the threads. Passing **`pajetrace=…`** records a timeline of *exactly that block* —
NGSolve writes a self-contained HTML **sunburst** (embedded below), and the **`Timers()`**
difference lists the hottest routines. We profile one `SolveProblem` solve.
*(Threads & tracing need a real OS — skipped in single-threaded JupyterLite.)*

In [ ]:
import glob, os, html, pathlib
from IPython.display import display, HTML

if sys.platform != "emscripten":                     # threads/tracing unavailable in JupyterLite
    SetNumThreads(4)
    before = {t["name"]: t["time"] for t in Timers()}    # snapshot to scope the timers
    with TaskManager(pajetrace=10**8):                   # the sunburst covers exactly this block
        SolveProblem("multigrid", levels=5)
    traces = sorted(glob.glob("ng*.html"), key=os.path.getmtime)   # the viewer NGSolve just wrote
    if traces:
        doc = html.escape(pathlib.Path(traces[-1]).read_text(), quote=True)
        display(HTML(f'<iframe srcdoc="{doc}" sandbox="allow-scripts" width="100%" height="460" '
                     f'style="border:1px solid #ddd;border-radius:8px" title="pajetrace sunburst"></iframe>'))
    delta = sorted(((t["time"] - before.get(t["name"], 0.0), t["name"]) for t in Timers()), reverse=True)
    print("hottest routines while solving:")
    for dt, name in delta[:5]:
        if dt > 0:
            print(f"  {dt * 1e3:8.2f} ms  {name[:46]}")
else:
    print("(threaded profiling / pajetrace needs a real OS — run locally or on Colab)")

That closes **Part I**: you can give the Beast geometry, functions, spaces, weak forms
and now the solvers to crack them at scale. Time to climb on — **into the saddle**, and
the features that real applications need.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("05-solving", "5 · First linear solve")
    _next = ("07-saddle-point", "7 · Mixed problems — the saddle point 🐎")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))